# Scoring de Crédit — Évaluation du Risque Bancaire
## Étape 2 : Construction du Modèle Prédictif

Après avoir exploré les données, je passe à la phase de modélisation. Mon objectif est d'entraîner un algorithme capable de distinguer automatiquement les clients fiables des clients à risque.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, classification_report
from sklearn.impute import SimpleImputer
import shap
import matplotlib.pyplot as plt

# J'active SHAP pour l'interprétabilité
shap.initjs()

### 1. Préparation et Fusion des données
Je fusionne les données de la demande actuelle avec le résumé de l'historique du bureau de crédit.

In [ ]:
app_train = pd.read_csv("../data/application_train.csv")
bureau = pd.read_csv("../data/bureau.csv")

# Agrégation du bureau
bureau_agg = bureau.groupby('SK_ID_CURR').size().reset_index(name='NB_PREV_CREDITS')

# Jointure
df = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')
df['NB_PREV_CREDITS'] = df['NB_PREV_CREDITS'].fillna(0)

print(f"Dimensions après fusion : {df.shape}")

### 2. Nettoyage et Encodage
Je transforme les variables textuelles en chiffres et je gère les valeurs manquantes.

In [ ]:
# Encodage simple des variables catégorielles
df = pd.get_dummies(df)

# Séparation des features et de la cible
X = df.drop(['TARGET', 'SK_ID_CURR'], axis=1)
y = df['TARGET']

# Imputation par la médiane pour les valeurs manquantes
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

### 3. Entraînement du Modèle
J'utilise un Random Forest avec une pondération équilibrée pour gérer le faible nombre de défauts.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

### 4. Évaluation
Je mesure la performance de mon modèle sur les données de test.

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

print(f"AUC-ROC : {roc_auc_score(y_test, y_prob):.3f}")
print(f"AUC-PR  : {average_precision_score(y_test, y_prob):.3f}")
print(f"Recall  : {recall_score(y_test, y_pred):.3f}")

print("\nTableau de classification :")
print(classification_report(y_test, y_pred))

### 5. Importance des variables et SHAP
Je cherche à comprendre quelles variables influencent le plus les décisions de mon modèle.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.nlargest(10).plot(kind='barh')
plt.title("Top 10 Feature Importance")
plt.show()

# Analyse SHAP sur un échantillon pour gagner du temps
explainer = shap.TreeExplainer(model)
X_sample = X_test.sample(100, random_state=42)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values[1], X_sample, plot_type="dot")

### 6. Sauvegarde
Je sauvegarde le modèle et la liste des colonnes pour l'application Streamlit.

In [ ]:
joblib.dump(model, "../app/model.joblib")
joblib.dump(X.columns.tolist(), "../app/model_columns.joblib")
print("Modèle et colonnes sauvegardés dans le dossier app/")

## Conclusions Business

1. **Performance Robuste** : Le modèle atteint un AUC-ROC supérieur à 0.70, ce qui est très encourageant pour un premier jet sur un problème aussi complexe.
2. **Capacité de détection** : Grâce à l'ajustement des poids, je parviens à identifier une partie significative des clients à risque, même s'ils sont rares.
3. **Transparence** : Les variables externes et l'âge restent les facteurs dominants. Le nombre de crédits précédents apporte également une nuance intéressante sur la stabilité financière du client.